ClimateGuard AI - Feature Engineering 

# ClimateGuard AI
## Notebook 03 – Advanced Feature Engineering

This notebook performs comprehensive feature engineering on the cleaned weather dataset.

### Objectives

- Load cleaned weather dataset
- Engineer temporal features
- Engineer weather-derived features
- Engineer geospatial features
- Engineer statistical features
- Engineer forecasting features
- Prepare the dataset for machine learning
- Save engineered dataset

# Imports 

In [21]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import time
import importlib

import numpy as np
import pandas as pd

from IPython.display import display

# Project root
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.feature_engineering import weather_features

weather_features = importlib.reload(weather_features)
WeatherFeatureEngineer = weather_features.WeatherFeatureEngineer

print("Imports completed.")

Imports completed.


# Load Cleaned dataset 

In [22]:
DATA_PATH = "../data/processed/03_feature_reduced_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Shape : {df.shape}")

display(df.head())

Dataset loaded successfully.
Shape : (119398, 35)


,country,location_name,region,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,India,Ashoknagar,Madhya Pradesh,24.57,77.72,Asia/Kolkata,1693286100,29-08-23 10:45,27.5,Partly cloudy,...,12.6,18.5,1,2,5:59 AM,6:41 PM,5:42 PM,3:38 AM,Waxing Gibbous,93
1,India,Raisen,Madhya Pradesh,23.33,77.80,Asia/Kolkata,1693286100,29-08-23 10:45,27.5,Sunny,...,10.7,14.2,1,1,6:00 AM,6:40 PM,5:39 PM,3:41 AM,Waxing Gibbous,93
2,India,Chhindwara,Madhya Pradesh,22.07,78.93,Asia/Kolkata,1693286100,29-08-23 10:45,26.3,Partly cloudy,...,16.8,20.7,2,2,5:56 AM,6:34 PM,5:32 PM,3:39 AM,Waxing Gibbous,93
3,India,Betul,Madhya Pradesh,21.86,77.93,Asia/Kolkata,1693286100,29-08-23 10:45,25.6,Cloudy,...,4.9,6.6,1,1,6:00 AM,6:38 PM,5:36 PM,3:43 AM,Waxing Gibbous,93
4,India,Hoshangabad,Madhya Pradesh,22.75,77.72,Asia/Kolkata,1693286100,29-08-23 10:45,27.2,Cloudy,...,11.4,14.8,1,1,6:01 AM,6:39 PM,5:38 PM,3:42 AM,Waxing Gibbous,93


## 4. Initialize Feature Engineer 

In [23]:
engineer = WeatherFeatureEngineer(df)
print("Feature engineer initialized")

Feature engineer initialized


## 5. Add Time-Based Features 

In [24]:
# Check for date column
date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower() or 'last_updated' in col.lower()]
print(f"Date columns found: {date_cols}")

if date_cols:
    # Check for date column
    date_cols = [
        col for col in df.columns
        if ('date' in col.lower()
            or 'datetime' in col.lower()
            or 'last_updated' in col.lower()
            or ('time' in col.lower() and col.lower() not in ['timezone', 'time_zone']))
    ]
    print(f"Date columns found: {date_cols}")

    if date_cols:
        if 'last_updated' in date_cols:
            datetime_col = 'last_updated'
        elif 'last_updated_epoch' in date_cols:
            datetime_col = 'last_updated_epoch'
        else:
            datetime_col = date_cols[0]

        if datetime_col == 'last_updated_epoch':
            df['last_updated'] = pd.to_datetime(df['last_updated_epoch'], unit='s')
            datetime_col = 'last_updated'
        else:
            df['last_updated'] = pd.to_datetime(df[datetime_col], format='%d-%m-%y %H:%M', errors='coerce')
            datetime_col = 'last_updated'

        engineer.add_time_features(datetime_col=datetime_col)
        print(f"Time features added. New columns: {[col for col in engineer.df.columns if col in ['year', 'month', 'day', 'hour', 'season', 'is_monthon']][:5]}")
    else:
        print("No date column found, skipping time features")
    print(f"Time features added. New columns: {[col for col in engineer.df.columns if col in ['year', 'month', 'day', 'hour', 'season', 'is_monthon']][:5]}")
else:
    print("No date column found, skipping time features")

Date columns found: ['timezone', 'last_updated_epoch', 'last_updated']
Date columns found: ['last_updated_epoch', 'last_updated']
Time features added. New columns: ['year', 'month', 'day', 'hour']
Time features added. New columns: ['year', 'month', 'day', 'hour']


## 6. Add Temperature Features 

In [25]:
engineer.add_temperature_features()
print("Temperature features added")
print(f"New columns: {[col for col in engineer.df.columns if 'temp' in col.lower() or 'heat' in col.lower()]}")

Temperature features added
New columns: ['temperature_celsius']


## 7. Add Rainfall Features 

In [26]:
engineer.add_rainfall_features()
print("Rainfall features added")
print(f"New columns: {[col for col in engineer.df.columns if 'rain' in col.lower() or 'precip' in col.lower()]}")

Rainfall features added
New columns: ['precip_mm']


## 8. Add Wind Features 

In [27]:
engineer.add_wind_features()
print("Wind features added")
print(f"New columns: {[col for col in engineer.df.columns if 'wind' in col.lower()]}")

Wind features added
New columns: ['wind_kph', 'wind_degree', 'wind_direction']


## 9. Add Pressure Features 

In [28]:
engineer.add_pressure_features()
print("Pressure features added")
print(f"New columns: {[col for col in engineer.df.columns if 'pressure' in col.lower()]}")

Pressure features added
New columns: ['pressure_mb']


## 10. Add Humidity Features 

In [29]:
engineer.add_humidity_features()
print("Humidity features added")
print(f"New columns: {[col for col in engineer.df.columns if 'humid' in col.lower()]}")

INFO:src.feature_engineering.weather_features:Added humidity features


Humidity features added
New columns: ['humidity', 'humidity_category']


## 11. Add UV Features 

In [30]:
engineer.add_uv_features()
print("UV features added")
print(f"New columns: {[col for col in engineer.df.columns if 'uv' in col.lower()]}")

INFO:src.feature_engineering.weather_features:Added UV features


UV features added
New columns: ['uv_index', 'uv_category', 'uv_alert']


## 12. Add Visibility Features 

In [31]:
engineer.add_visibility_features()
print("Visibility features added")
print(f"New columns: {[col for col in engineer.df.columns if 'visib' in col.lower()]}")

Visibility features added
New columns: ['visibility_km']


## 13. Add Cloud Features 

In [32]:
engineer.add_cloud_features()
print("Cloud features added")
print(f"New columns: {[col for col in engineer.df.columns if 'cloud' in col.lower()]}")

Cloud features added
New columns: ['cloud']


## 14. Add Air Quality Features 

In [33]:
engineer.add_air_quality_features()
print("Air quality features added")
print(f"New columns: {[col for col in engineer.df.columns if 'pm' in col.lower() or 'aqi' in col.lower() or 'pollution' in col.lower()]}")

Air quality features added
New columns: ['air_quality_PM2.5', 'air_quality_PM10']


## 15. Add Geographic Features 

In [34]:
# Check for latitude/longitude columns
lat_cols = [col for col in engineer.df.columns if 'lat' in col.lower()]
lon_cols = [col for col in engineer.df.columns if 'lon' in col.lower()]

if lat_cols and lon_cols:
    engineer.add_geographic_features(lat_cols[0], lon_cols[0])
    print("Geographic features added")
    print(f"New columns: {[col for col in engineer.df.columns if 'geo' in col.lower() or 'band' in col.lower() or 'climate_zone' in col.lower()]}")
else:
    print("Latitude/Longitude columns not found")

INFO:src.feature_engineering.weather_features:Added geographic features


Geographic features added
New columns: ['latitude_band', 'longitude_band', 'climate_zone']


## 16. Add Advanced Engineered Features 

Now we add rolling statistics, trend and lag features, interactions, derived weather metrics, and forecasting signals.

In [35]:
# import importlib
# import time
# from src.feature_engineering import weather_features

# # Reload the feature-engineering module in case the kernel has a stale import.
# weather_features = importlib.reload(weather_features)
# WeatherFeatureEngineer = weather_features.WeatherFeatureEngineer
# engineer = WeatherFeatureEngineer(engineer.df)

# # Optimized single-run advanced features (no duplicates)
# start = time.time()
# try:
#     engineer.add_statistical_features(windows=[3,6,12], quantiles=[])
# except TypeError:
#     engineer.add_statistical_features(windows=[3,6,12], quantiles=[])
# print(f"Statistical features done: {time.time()-start:.2f}s")
# start = time.time()
# engineer.add_expanding_features()
# print(f"Expanding features done: {time.time()-start:.2f}s")
# start = time.time()
# engineer.add_exponential_features(spans=[3,6])
# print(f"Exponential features done: {time.time()-start:.2f}s")
# start = time.time()
# engineer.add_lag_features(lags=[1,3,6])
# print(f"Lag features done: {time.time()-start:.2f}s")
# start = time.time()
# engineer.add_trend_features(windows=[3,6])
# print(f"Trend features done: {time.time()-start:.2f}s")
# start = time.time()
# engineer.add_interaction_features()
# print(f"Interaction features done: {time.time()-start:.2f}s")
# start = time.time()
# engineer.add_derived_weather_features()
# print(f"Derived weather features done: {time.time()-start:.2f}s")
# target_col = "precipitation" if "precipitation" in engineer.df.columns else None
# start = time.time()
# engineer.prepare_forecasting_features(target_col=target_col, future_horizon=3, lags=[1,3], windows=[3], spans=[3])
# print(f"Forecasting features done: {time.time()-start:.2f}s")
# print("Added advanced engineered features (optimized single run).")
# print(f"Forecast target column: {target_col}")


In [36]:
import gc
import time
import importlib

from src.feature_engineering import weather_features

# Reload module
weather_features = importlib.reload(weather_features)
WeatherFeatureEngineer = weather_features.WeatherFeatureEngineer

# ----------------------------------------------------
# Reuse existing dataframe (NO extra copy)
# ----------------------------------------------------
df = engineer.df

# Remove duplicate columns
df = df.loc[:, ~df.columns.duplicated()]

engineer = WeatherFeatureEngineer(df)

# Use only original numeric columns
base_cols = df.select_dtypes(include="number").columns.tolist()

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")
print(f"Numeric : {len(base_cols)}")

# ====================================================
# SAFE FEATURE ENGINEERING
# ====================================================

# 1. Time Features
if not hasattr(engineer, "timestamp_col"):
    try:
        engineer.add_time_features()
    except Exception:
        pass

gc.collect()

# ----------------------------------------------------

# 2. Weather Physics
start = time.time()

engineer.add_derived_weather_features()

gc.collect()

print(f"Derived weather : {time.time()-start:.2f}s")

# ----------------------------------------------------

# 3. Risk Features
start = time.time()

engineer.add_risk_features()

gc.collect()

print(f"Risk features : {time.time()-start:.2f}s")

# ----------------------------------------------------

# 4. Interaction Features
start = time.time()

engineer.add_interaction_features()

gc.collect()

print(f"Interaction : {time.time()-start:.2f}s")

# ----------------------------------------------------

# 5. Very Small Lag Features
start = time.time()

engineer.add_lag_features(
    columns=base_cols,
    lags=[1]
)

gc.collect()

print(f"Lag : {time.time()-start:.2f}s")

# ----------------------------------------------------

# 6. Small Rolling Mean
start = time.time()

for col in base_cols:
    engineer.df[f"{col}_roll_mean3"] = (
        engineer.df[col]
        .rolling(window=3, min_periods=1)
        .mean()
    )

gc.collect()

print(f"Rolling Mean : {time.time()-start:.2f}s")

# ----------------------------------------------------

# 7. EMA ONLY (NO STD)
start = time.time()

for col in base_cols:
    engineer.df[f"{col}_ema3"] = (
        engineer.df[col]
        .ewm(span=3, adjust=False)
        .mean()
    )

gc.collect()

print(f"EMA : {time.time()-start:.2f}s")

# ----------------------------------------------------

print("\nFinal Shape")
print(engineer.df.shape)

print("\nSAFE FEATURE ENGINEERING COMPLETED")

Rows    : 119,398
Columns : 84
Numeric : 62


Derived weather : 0.18s
Risk features : 0.15s
Interaction : 0.14s
Lag : 0.17s
Rolling Mean : 0.38s
EMA : 0.48s

Final Shape
(119398, 272)

SAFE FEATURE ENGINEERING COMPLETED


## 17. Feature Selection and Review 

Review the full engineered feature set and ensure the forecast columns are available if the target exists.

In [37]:
engineered_df = engineer.get_engineered_data()
print(f"Engineered data shape after advanced features: {engineered_df.shape}")
print(f"New columns added: {len(engineered_df.columns) - len(df.columns)}")
print("Sample engineered columns:")
print(engineered_df.columns.tolist()[:40])

Engineered data shape after advanced features: (119398, 272)
New columns added: 188
Sample engineered columns:
['country', 'location_name', 'region', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'last_updated', 'temperature_celsius', 'condition_text', 'wind_kph', 'wind_degree', 'wind_direction', 'pressure_mb', 'precip_mm', 'humidity', 'cloud', 'feels_like_celsius', 'visibility_km', 'uv_index', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination', 'year', 'month', 'day', 'hour', 'minute']


## 18. View Engineered Data  

In [38]:
engineered_df = engineer.get_engineered_data()
print(f"Final engineered data shape: {engineered_df.shape}")
print(f"\nTotal columns: {len(engineered_df.columns)}")
print(f"\nFirst few rows:")
display(engineered_df.head())

Final engineered data shape: (119398, 272)

Total columns: 272

First few rows:


,country,location_name,region,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,condition_text,...,month_cos_ema3,day_of_year_sin_ema3,day_of_year_cos_ema3,week_of_year_sin_ema3,week_of_year_cos_ema3,hour_sin_ema3,hour_cos_ema3,minute_sin_ema3,minute_cos_ema3,uv_alert_ema3
0,India,Kupwara,Jammu and Kashmir,34.03,74.26,Asia/Karachi,1693525500,2023-01-09 04:45:00,14.6,Clear,...,0.866025,0.154309,0.988023,0.239316,0.970942,0.866025,0.500000,-1.000,-1.836970e-16,0.0
1,India,Betul,Madhya Pradesh,21.86,77.93,Asia/Kolkata,1693525500,2023-01-09 05:15:00,22.8,Partly cloudy,...,0.866025,0.154309,0.988023,0.239316,0.970942,0.915976,0.379410,0.000,4.978996e-17,0.0
2,India,Hoshangabad,Madhya Pradesh,22.75,77.72,Asia/Kolkata,1693525500,2023-01-09 05:15:00,24.1,Partly cloudy,...,0.866025,0.154309,0.988023,0.239316,0.970942,0.940951,0.319114,0.500,1.665335e-16,0.0
3,India,Sehore,Madhya Pradesh,23.20,77.08,Asia/Kolkata,1693525500,2023-01-09 05:15:00,23.4,Clear,...,0.866025,0.154309,0.988023,0.239316,0.970942,0.953438,0.288967,0.750,2.249052e-16,0.0
4,India,Jabalpur,Madhya Pradesh,23.17,79.95,Asia/Kolkata,1693525500,2023-01-09 05:15:00,23.4,Partly cloudy,...,0.866025,0.154309,0.988023,0.239316,0.970942,0.959682,0.273893,0.875,2.540911e-16,0.0


## 19. Feature Summary   

In [39]:
print("\nFeature Summary:")
print(f"Original columns: {len(df.columns)}")
print(f"Engineered columns: {len(engineered_df.columns)}")
print(f"New features added: {len(engineered_df.columns) - len(df.columns)}")

print("\nAll columns:")
for i, col in enumerate(engineered_df.columns, 1):
    print(f"{i}. {col}")


Feature Summary:
Original columns: 84
Engineered columns: 272
New features added: 188

All columns:
1. country
2. location_name
3. region
4. latitude
5. longitude
6. timezone
7. last_updated_epoch
8. last_updated
9. temperature_celsius
10. condition_text
11. wind_kph
12. wind_degree
13. wind_direction
14. pressure_mb
15. precip_mm
16. humidity
17. cloud
18. feels_like_celsius
19. visibility_km
20. uv_index
21. gust_kph
22. air_quality_Carbon_Monoxide
23. air_quality_Ozone
24. air_quality_Nitrogen_dioxide
25. air_quality_Sulphur_dioxide
26. air_quality_PM2.5
27. air_quality_PM10
28. air_quality_us-epa-index
29. air_quality_gb-defra-index
30. sunrise
31. sunset
32. moonrise
33. moonset
34. moon_phase
35. moon_illumination
36. year
37. month
38. day
39. hour
40. minute
41. second
42. day_of_week
43. day_name
44. month_name
45. week_of_year
46. quarter
47. day_of_year
48. days_in_month
49. is_weekend
50. is_weekday
51. is_month_start
52. is_month_end
53. is_quarter_start
54. is_quarter_en

## 20. Save Engineered Data  

In [40]:
# Save engineered data
df.to_csv('../data/processed/04_engineered_data.csv', index=False)
print("\nEngineered data saved to ../data/processed/04_engineered_data.csv")


Engineered data saved to ../data/processed/04_engineered_data.csv


## Summary
 
In this notebook, we:
1. Loaded the cleaned weather data
2. Renamed columns to match feature engineer expectations
3. Added time-based features (hour, day, month, season)
4. Added temperature features (heat index, wind chill, temperature ranges)
5. Added rainfall features (rain intensity, dry periods)
6. Added wind features (wind direction categories, gust ratios)
7. Added pressure features (pressure changes, pressure categories)
8. Added humidity features (dew point, comfort index)
9. Added UV features (UV categories, UV risk levels)
10. Added visibility features (visibility categories)
11. Added cloud features (cloud cover categories)
12. Added air quality features (AQI categories, pollutant ratios)
13. Added geographic features (climate bands)
14. Added advanced engineered features (statistical, lag, trend, interaction, derived weather, forecasting)
15. Saved the engineered data for EDA

Next: Exploratory Data Analysis